# ComfyUI on Colab → 外部公開セットアップ

このノートブックは、Google Colab上にComfyUIをセットアップし、`cloudflared`で外部からアクセス可能な公開URLを発行します。
発行されたURLを、手元の `app/index.html`（Comfy Simple Studio）の「バックエンドURL」欄に貼り付ければ使えます。

**手順**
1. ランタイムタイプを GPU に変更（ランタイム → ランタイムのタイプを変更 → T4以上）
2. 上から順に全セルを実行
3. 最後のセルに表示される `https://xxxx.trycloudflare.com` をコピー
4. Colabは一定時間操作がないと切断されます。切断されたら最後のセルを再実行（URLは変わります）してください。

In [ ]:
# 1. ComfyUI を取得して依存ライブラリをインストール
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -r requirements.txt -q

In [ ]:
# 2. ベースチェックポイントモデルをダウンロード
# 他のモデルを使いたい場合は URL と保存先ファイル名を変えてください（models/checkpoints/ 以下に .safetensors を置く）。
!wget -q --show-progress -O models/checkpoints/sd_v1-5.safetensors \
  https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors
print('ダウンロード完了')

In [ ]:
# 3. cloudflared (トンネル用バイナリ) を取得
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('cloudflared 準備完了')

In [ ]:
# 4. ComfyUI を起動し、cloudflared で外部公開 URL を発行
# --enable-cors-header は、別オリジン（ブラウザで開いている app/index.html）からの fetch を許可するために必須です。
import subprocess, time, re

comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--enable-cors-header'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# ComfyUI が起動するまで少し待つ
time.sleep(15)

tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

print('公開URLを取得中...')
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if m:
        print('\n==================================')
        print(' ComfyUI 公開URL:', m.group(0))
        print(' → app/index.html の「バックエンドURL」欄に貼り付けてください')
        print('==================================\n')
        break
print('ComfyUI と cloudflared はバックグラウンドで実行中です。このセルの実行は終了しても問題ありません。')

## トラブルシューティング
- URLが表示されない場合: 上のセルをもう一度実行してください（トンネル接続に数秒かかることがあります）。
- アプリ側で「接続失敗」になる場合: URLの該写し(末尾の / など)を確認してください。
- 一定時間経つとColabセッションが切断されます。その場合は最初から全セルをやり直してください（URLが変わるのでアプリ側も更新）。